In [151]:
# !pip install langchain-groq

In [152]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
import operator
from langgraph.checkpoint.memory import InMemorySaver

In [153]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [154]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY
)

In [155]:
class JokeState(TypedDict):

  topic: str
  joke: str
  explanation: str

In [156]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [157]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [158]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [159]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'cricket'}, config=config1)

{'topic': 'cricket',
 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the batsman was *always* hitting high scores!',
 'explanation': '**The joke**\n\n> *Why did the cricket team bring a ladder to the match?*  \n> *Because they heard the batsman was **always** hitting high scores!*\n\n---\n\n### 1. The literal set‑up\n\n- **Cricket** is a bat‑and‑ball sport.  \n- The **batsman** is the player who faces the bowler and tries to hit the ball with his bat.  \n- A **score** in cricket is the number of runs a batsman (or the whole team) accumulates.  \n- When a batsman “hits high scores,” we normally mean he scores a lot of runs—e.g., 50, 100, 150 runs in an innings.\n\n---\n\n### 2. The wordplay (the pun)\n\nThe humor comes from treating the phrase **“high scores”** in two different ways at the same time:\n\n| Meaning of “high scores” | What it really means in cricket | What the joke pretends it could mean |\n|--------------------------|-----------------

In [160]:
print(workflow.get_state(config1))

StateSnapshot(values={'topic': 'cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the batsman was *always* hitting high scores!', 'explanation': '**The joke**\n\n> *Why did the cricket team bring a ladder to the match?*  \n> *Because they heard the batsman was **always** hitting high scores!*\n\n---\n\n### 1. The literal set‑up\n\n- **Cricket** is a bat‑and‑ball sport.  \n- The **batsman** is the player who faces the bowler and tries to hit the ball with his bat.  \n- A **score** in cricket is the number of runs a batsman (or the whole team) accumulates.  \n- When a batsman “hits high scores,” we normally mean he scores a lot of runs—e.g., 50, 100, 150 runs in an innings.\n\n---\n\n### 2. The wordplay (the pun)\n\nThe humor comes from treating the phrase **“high scores”** in two different ways at the same time:\n\n| Meaning of “high scores” | What it really means in cricket | What the joke pretends it could mean |\n|-------------------------

In [161]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the batsman was *always* hitting high scores!', 'explanation': '**The joke**\n\n> *Why did the cricket team bring a ladder to the match?*  \n> *Because they heard the batsman was **always** hitting high scores!*\n\n---\n\n### 1. The literal set‑up\n\n- **Cricket** is a bat‑and‑ball sport.  \n- The **batsman** is the player who faces the bowler and tries to hit the ball with his bat.  \n- A **score** in cricket is the number of runs a batsman (or the whole team) accumulates.  \n- When a batsman “hits high scores,” we normally mean he scores a lot of runs—e.g., 50, 100, 150 runs in an innings.\n\n---\n\n### 2. The wordplay (the pun)\n\nThe humor comes from treating the phrase **“high scores”** in two different ways at the same time:\n\n| Meaning of “high scores” | What it really means in cricket | What the joke pretends it could mean |\n|------------------------

In [162]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti get a promotion?\n\nBecause it always knows how to *pasta*‑ble! 🍝😄',
 'explanation': '**Joke:**  \n*Why did the spaghetti get a promotion?*  \n*Because it always knows how to **pasta‑ble**!* 🍝😄  \n\n---\n\n## 1. The literal set‑up  \n\n- **Spaghetti** is a type of pasta.  \n- In a typical workplace story, a “promotion” is earned when someone consistently does a good job.\n\nSo the listener expects an answer that explains why a piece of food could be “good at work.” The absurdity of treating spaghetti like an employee is already a source of humor (the **incongruity** principle).\n\n---\n\n## 2. The wordplay (the punchline)\n\n### a. “pasta‑ble” ≈ “passable”\n\n- **Passable** (pronounced /ˈpæsəbəl/) means *acceptable* or *good enough*—the kind of performance that might earn a promotion.  \n- By swapping the “s” in *passable* for the word **pasta**, we get **pasta‑ble** (pronounced the same way).  \n\nThus the phrase *“knows how to pasta‑

In [163]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti get a promotion?\n\nBecause it always knows how to *pasta*‑ble! 🍝😄', 'explanation': '**Joke:**  \n*Why did the spaghetti get a promotion?*  \n*Because it always knows how to **pasta‑ble**!* 🍝😄  \n\n---\n\n## 1. The literal set‑up  \n\n- **Spaghetti** is a type of pasta.  \n- In a typical workplace story, a “promotion” is earned when someone consistently does a good job.\n\nSo the listener expects an answer that explains why a piece of food could be “good at work.” The absurdity of treating spaghetti like an employee is already a source of humor (the **incongruity** principle).\n\n---\n\n## 2. The wordplay (the punchline)\n\n### a. “pasta‑ble” ≈ “passable”\n\n- **Passable** (pronounced /ˈpæsəbəl/) means *acceptable* or *good enough*—the kind of performance that might earn a promotion.  \n- By swapping the “s” in *passable* for the word **pasta**, we get **pasta‑ble** (pronounced the same way).  \n\nThus the phrase *“

In [164]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti get a promotion?\n\nBecause it always knows how to *pasta*‑ble! 🍝😄', 'explanation': '**Joke:**  \n*Why did the spaghetti get a promotion?*  \n*Because it always knows how to **pasta‑ble**!* 🍝😄  \n\n---\n\n## 1. The literal set‑up  \n\n- **Spaghetti** is a type of pasta.  \n- In a typical workplace story, a “promotion” is earned when someone consistently does a good job.\n\nSo the listener expects an answer that explains why a piece of food could be “good at work.” The absurdity of treating spaghetti like an employee is already a source of humor (the **incongruity** principle).\n\n---\n\n## 2. The wordplay (the punchline)\n\n### a. “pasta‑ble” ≈ “passable”\n\n- **Passable** (pronounced /ˈpæsəbəl/) means *acceptable* or *good enough*—the kind of performance that might earn a promotion.  \n- By swapping the “s” in *passable* for the word **pasta**, we get **pasta‑ble** (pronounced the same way).  \n\nThus the phrase *

### Time Travel

In [172]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f15b136-ab59-6658-bfff-63aa747191f6"}})

StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f15b136-ab59-6658-bfff-63aa747191f6'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-05-29T04:03:57.798755+00:00', parent_config=None, tasks=(PregelTask(id='0c9c1265-39f3-3ac0-7334-cd841f1a1f7b', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'topic': 'cricket'}),), interrupts=())

In [173]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f15b136-ab59-6658-bfff-63aa747191f6"}})

{'topic': 'cricket',
 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the scores were *up* there and wanted to *reach* a new **run**‑level! 🏏😄',
 'explanation': '**The joke broken down**\n\n| Part of the joke | What it literally says | The word‑play / cricket reference |\n|------------------|------------------------|-----------------------------------|\n| “Why did the cricket team bring a **ladder** to the match?” | A ladder is a piece of equipment used for climbing. | In cricket there is no real reason to bring a ladder, so the set‑up already feels absurd. |\n| “Because they heard the **scores were *up* there**” | “Up there” can mean “high up in the sky” (where a ladder would help you reach something). | In sport, “the scores are up” means the total runs or points are high. The joke pretends the team thinks the score is physically located somewhere high, like on a wall or a scoreboard that’s out of reach. |\n| “and wanted to **reach** a new **run‑l

In [175]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the scores were *up* there and wanted to *reach* a new **run**‑level! 🏏😄', 'explanation': '**The joke broken down**\n\n| Part of the joke | What it literally says | The word‑play / cricket reference |\n|------------------|------------------------|-----------------------------------|\n| “Why did the cricket team bring a **ladder** to the match?” | A ladder is a piece of equipment used for climbing. | In cricket there is no real reason to bring a ladder, so the set‑up already feels absurd. |\n| “Because they heard the **scores were *up* there**” | “Up there” can mean “high up in the sky” (where a ladder would help you reach something). | In sport, “the scores are up” means the total runs or points are high. The joke pretends the team thinks the score is physically located somewhere high, like on a wall or a scoreboard that’s out of reach. |\n| “and wanted to **r

### Updating State

In [176]:
workflow.update_state(
    {
        "configurable": {
            "thread_id": "1",
            "checkpoint_ns": "",
            "checkpoint_id": "1f15b136-ab59-6658-bfff-63aa747191f6"
        }
    },
    {"topic": "samosa"}
)

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f15b13e-b59b-6928-8000-e76005af2d53'}}

In [177]:
print(workflow.get_state(config1))

StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b13e-b59b-6928-8000-e76005af2d53'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-05-29T04:07:33.622794+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b136-ab59-6658-bfff-63aa747191f6'}}, tasks=(PregelTask(id='377ed742-0c78-d503-4885-1027e66c517f', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=())


In [178]:
result = workflow.invoke(None, config1)
print(result)

{'topic': 'samosa', 'joke': 'Why did the samosa apply for a job?\n\nBecause it heard the position was *filled* with lots of “crunch” time and it wanted to prove it could handle the *filling* responsibilities!', 'explanation': '**What makes the joke funny?**  \n\nThe humor comes from a series of word‑plays that link the literal qualities of a samosa (a fried, crunchy, stuffed snack) with the figurative language we use when we talk about jobs. Let’s break it down step by step.\n\n| Part of the joke | Literal meaning (samosa) | Figurative meaning (job) | Why it works |\n|------------------|--------------------------|--------------------------|--------------|\n| **“Why did the samosa apply for a job?”** | Sets up the expectation that a food item is behaving like a person. | Introduces a classic “Why did X do Y?” set‑up that primes the listener for a punchline. | The absurdity of a samosa being a job‑seeker creates the comedic tension. |\n| **“Because it heard the position was *filled* …”**

In [179]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa apply for a job?\n\nBecause it heard the position was *filled* with lots of “crunch” time and it wanted to prove it could handle the *filling* responsibilities!', 'explanation': '**What makes the joke funny?**  \n\nThe humor comes from a series of word‑plays that link the literal qualities of a samosa (a fried, crunchy, stuffed snack) with the figurative language we use when we talk about jobs. Let’s break it down step by step.\n\n| Part of the joke | Literal meaning (samosa) | Figurative meaning (job) | Why it works |\n|------------------|--------------------------|--------------------------|--------------|\n| **“Why did the samosa apply for a job?”** | Sets up the expectation that a food item is behaving like a person. | Introduces a classic “Why did X do Y?” set‑up that primes the listener for a punchline. | The absurdity of a samosa being a job‑seeker creates the comedic tension. |\n| **“Because it heard the posi

### Fault Tolerance

In [52]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [53]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [54]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(60)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [55]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [56]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [57]:
graph.get_state({"configurable": {"thread_id": 'thread-1'}})

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b0cb-7418-665e-8001-bb8d1f3f94ac'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-05-29T03:15:59.745573+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b0cb-7413-6f36-8000-8b016506d814'}}, tasks=(PregelTask(id='15d203c9-21bb-6b1e-2c30-4e5ea5bcd29f', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [58]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b0cb-7418-665e-8001-bb8d1f3f94ac'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-05-29T03:15:59.745573+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b0cb-7413-6f36-8000-8b016506d814'}}, tasks=(PregelTask(id='15d203c9-21bb-6b1e-2c30-4e5ea5bcd29f', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b0cb-7413-6f36-8000-8b016506d814'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-05-29T03:15:59.743753+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'ch

In [59]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [61]:
graph.get_state({"configurable": {"thread_id": 'thread-1'}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b0ce-34f8-6fda-8003-b166c9e6e034'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-05-29T03:17:13.657319+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b0ce-34f5-6345-8002-caed15f2413a'}}, tasks=(), interrupts=())